### Job Documentation - Full Refresh Pipeline

This document describes how the **Databricks Job** was configured to orchestrate the full Formula 1 lakehouse pipeline (bronze -> silver -> gold) as a single scheduled workflow.

---

#### Job Details

| Setting | Value |
| --- | --- |
| **Job Name** | `jobs_formula1_lakehouse_full_refresh` |
| **Job ID** | `505642093024219` |
| **Trigger** | Manual (on-demand) |
| **Max Concurrent Runs** | 1 |
| **Failure Notification** | Email to `arthisararaj@gmail.com` |

---

#### Compute Configuration

| Setting | Value |
| --- | --- |
| **Cluster Type** | Job Cluster (spins up per run, shuts down after) |
| **Node Type** | `Standard_D4ds_v4` |
| **Workers** | 0 (single-node mode) |
| **Cluster Key** | `Job_cluster` |

All 17 tasks share the same job cluster - this means they reuse the same compute and don't spin up separate clusters per task.

---

#### Task Dependency Graph (17 Tasks)

The job follows the **medallion architecture** - tasks are ordered so each layer completes before the next begins.

##### Layer 1: Bronze (Ingestion) - No Dependencies
These 6 tasks run **in parallel** at the start since they have no upstream dependencies:

| Task Key | Notebook |
| --- | --- |
| `01_ingest_circuit_files` | `02-bronze/01) Ingest circuit files` |
| `02_ingest_races_files` | `02-bronze/02) Ingest race files` |
| `03_ingest_constructors_files` | `02-bronze/03) Ingest Constructor Files` |
| `04_ingest_drivers_files` | `02-bronze/04) Ingest Driver File` |
| `05_ingest_results_files` | `02-bronze/05) Ingest Results Files` |
| `06_ingest_sprints_files` | `02-bronze/06) Ingest Sprints Files` |

##### Layer 2: Silver (Transformation) - Depends on Bronze
Each silver task waits for its corresponding bronze task to finish:

| Task Key | Depends On | Notebook |
| --- | --- | --- |
| `01_Transform_circuits_data` | `01_ingest_circuit_files` | `03-silver/01) Transformation Circuits data` |
| `02_transform_races_files` | `02_ingest_races_files` | `03-silver/02) Transformation Races data` |
| `03_Transform_constructors_data` | `03_ingest_constructors_files` | `03-silver/03) Transformation Conductor data` |
| `04_Transform_drivers_data` | `04_ingest_drivers_files` | `03-silver/04) Transformation Drivers data` |
| `05_Transform_results_data` | `05_ingest_results_files` | `03-silver/05) Transform Results Data` |
| `06_Transform_sprints_data` | `06_ingest_sprints_files` | `03-silver/06) Transform Sprints Data` |

##### Layer 3: Gold Reference Table - No Dependencies
Runs in parallel with everything else since it's a static lookup:

| Task Key | Depends On | Notebook |
| --- | --- | --- |
| `01_Building_Nationality_Reference_table` | None | `04-gold/11) Building Nationality Reference table` |

##### Layer 4: Gold (Dimensions & Fact) - Depends on Silver + Reference
These run last, after their upstream silver tables and the reference table are ready:

| Task Key | Depends On | Notebook |
| --- | --- | --- |
| `01_Build_races_dimention` | `01_Transform_circuits_data`, `02_transform_races_files` | `04-gold/01) Build Races Dimention` |
| `02_Build_Constructors_Dimention` | `01_Building_Nationality_Reference_table`, `03_Transform_constructors_data` | `04-gold/02) Build Constructors Dimention` |
| `03_Build_Drivers_Dimention` | `01_Building_Nationality_Reference_table`, `04_Transform_drivers_data` | `04-gold/03) Build Drivers Dimention` |
| `04_Build_Session_Results_Fact` | `05_Transform_results_data`, `06_Transform_sprints_data` | `04-gold/04 Build Session Results Fact` |

---

#### How the DAG Flows

```
BRONZE (parallel)          SILVER (sequential)         GOLD (final)
─────────────────          ───────────────────         ────────────
circuits ──────────────> transform_circuits ─────┐
races ─────────────────> transform_races ────────┼──> dim_races
conductors ────────────> transform_constructors ─┼──> dim_constructors
drivers ───────────────> transform_drivers ──────┼──> dim_drivers
results ───────────────> transform_results ──────┼──> facts_session_results
sprints ───────────────> transform_sprints ──────┘
                                                 ┌──> dim_constructors
ref_nationality_regions (parallel) ──────────────┼──> dim_drivers
                                                 └
```

---

#### Recent Run History

| Run ID | Status | Duration | Trigger |
| --- | --- | --- | --- |
| 542281592034852 | SUCCESS | \~23 min | Manual |
| 925496195159419 | SUCCESS | \~8.5 min | Manual |
| 493519563084574 | SUCCESS | \~4.4 min | Manual |
| 978228999677548 | SUCCESS | \~7.4 min | Manual |

All recent runs completed successfully.

---

#### How to Run

1. Go to **Jobs** in the Databricks sidebar
2. Find `jobs_formula1_lakehouse_full_refresh`
3. Click **Run Now** to trigger a full refresh
4. Monitor progress in the run details page - tasks show green (success) or red (failed)
5. If a task fails, you'll receive an email notification

---

#### How This Was Set Up

1. Created a new job in **Jobs** > **Create Job**
2. Named it `jobs_formula1_lakehouse_full_refresh`
3. Added a shared **Job Cluster** (`Standard_D4ds_v4`, single-node) to keep costs low
4. Added 17 notebook tasks one by one, pointing each to its notebook path
5. Configured **dependencies** between tasks (bronze -> silver -> gold)
6. Set **email notification** on failure
7. Left trigger as manual (no schedule) - run on demand when new data arrives

#### Job YAML Config (for Databricks Asset Bundles)

This is the exported job definition in YAML format. You can place this in a `databricks.yml` or `resources/jobs.yml` file in your Git repo to recreate the job using `databricks bundle deploy`.

```yaml
resources:
  jobs:
    jobs_formula1_lakehouse_full_refresh:
      name: jobs_formula1_lakehouse_full_refresh
      email_notifications:
        on_failure:
          - arthisararaj@gmail.com
      tasks:
        - task_key: 01_Building_Nationality_Reference_table
          notebook_task:
            notebook_path: ./04-gold/11) Building Nationality Reference table
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 01_ingest_circuit_files
          notebook_task:
            notebook_path: ./02-bronze/01) Ingest circuit files
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 01_Transform_circuits_data
          depends_on:
            - task_key: 01_ingest_circuit_files
          notebook_task:
            notebook_path: ./03-silver/01) Transformation Circuits data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 02_ingest_races_files
          notebook_task:
            notebook_path: ./02-bronze/02) Ingest race files
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 02_transform_races_files
          depends_on:
            - task_key: 02_ingest_races_files
          notebook_task:
            notebook_path: ./03-silver/02) Transformation Races data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 01_Build_races_dimention
          depends_on:
            - task_key: 01_Transform_circuits_data
            - task_key: 02_transform_races_files
          notebook_task:
            notebook_path: ./04-gold/01) Build Races Dimention
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 03_ingest_constructors_files
          notebook_task:
            notebook_path: ./02-bronze/03) Ingest Constructor Files
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 03_Transform_constructors_data
          depends_on:
            - task_key: 03_ingest_constructors_files
          notebook_task:
            notebook_path: ./03-silver/03) Transformation Conductor data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 02_Build_Constructors_Dimention
          depends_on:
            - task_key: 01_Building_Nationality_Reference_table
            - task_key: 03_Transform_constructors_data
          notebook_task:
            notebook_path: ./04-gold/02) Build Constructors Dimention
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 04_ingest_drivers_files
          notebook_task:
            notebook_path: ./02-bronze/04) Ingest Driver File
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 04_Transform_drivers_data
          depends_on:
            - task_key: 04_ingest_drivers_files
          notebook_task:
            notebook_path: ./03-silver/04) Transformation Drivers data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 03_Build_Drivers_Dimention
          depends_on:
            - task_key: 04_Transform_drivers_data
            - task_key: 01_Building_Nationality_Reference_table
          notebook_task:
            notebook_path: ./04-gold/03) Build Drivers Dimention
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 05_ingest_results_files
          notebook_task:
            notebook_path: ./02-bronze/05) Ingest Results Files
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 04_Transform_results_data
          depends_on:
            - task_key: 05_ingest_results_files
          notebook_task:
            notebook_path: ./03-silver/05) Transform Results Data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 06_ingest_sprints_files
          notebook_task:
            notebook_path: ./02-bronze/06) Ingest Sprints Files
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 06_Transform_sprints_data
          depends_on:
            - task_key: 06_ingest_sprints_files
          notebook_task:
            notebook_path: ./03-silver/06) Transform Sprints Data
            source: WORKSPACE
          job_cluster_key: Job_cluster

        - task_key: 04_Build_Session_Results_Fact
          depends_on:
            - task_key: 05_ingest_results_files
            - task_key: 06_Transform_sprints_data
          notebook_task:
            notebook_path: ./04-gold/04 Build Session Results Fact
            source: WORKSPACE
          job_cluster_key: Job_cluster

      job_clusters:
        - job_cluster_key: Job_cluster
          new_cluster:
            spark_version: 17.3.x-scala2.13
            spark_conf:
              spark.master: "local[*, 4]"
              spark.databricks.cluster.profile: singleNode
            azure_attributes:
              first_on_demand: 1
              availability: SPOT_WITH_FALLBACK_AZURE
              spot_bid_max_price: -1
            node_type_id: Standard_D4ds_v4
            custom_tags:
              ResourceClass: SingleNode
            num_workers: 0
            enable_elastic_disk: true
            data_security_mode: SINGLE_USER
            runtime_engine: STANDARD
      queue:
        enabled: true
```

---

#### How to deploy this YAML to Databricks (step by step)

##### Step 1: Set up your GitHub repo folder structure
```
Formula1-Project/
├── databricks.yml            <-- bundle config (create this)
├── resources/
│   └── jobs.yml              <-- paste the YAML above here
├── 02-bronze/
│   └── (all bronze notebooks)
├── 03-silver/
│   └── (all silver notebooks)
├── 04-gold/
│   └── (all gold notebooks)
└── 00-common/
    └── (config + helpers)
```

##### Step 2: Create `databricks.yml` at the root
```yaml
bundle:
  name: formula1_lakehouse

workspace:
  host: https://<your-databricks-workspace-url>

include:
  - resources/*.yml
```
Replace `<your-databricks-workspace-url>` with your actual workspace URL (e.g., `https://adb-xxxx.azuredatabricks.net`).

##### Step 3: Install the Databricks CLI (if not already)
```bash
pip install databricks-cli
```
Or download from [Databricks CLI docs](https://docs.databricks.com/dev-tools/cli/install.html).

##### Step 4: Authenticate
```bash
databricks configure --token
```
Enter your workspace URL and a personal access token (generate one in **User Settings** > **Developer** > **Access Tokens**).

##### Step 5: Deploy the job
```bash
cd Formula1-Project/
databricks bundle deploy
```
This reads the YAML and creates (or updates) the job in your Databricks workspace.

##### Step 6: Run the job from the CLI (optional)
```bash
databricks bundle run jobs_formula1_lakehouse_full_refresh
```
Or just go to **Jobs** in the Databricks UI and click **Run Now**.

---

##### Quick summary
| What | Command |
| --- | --- |
| Deploy job to Databricks | `databricks bundle deploy` |
| Run the job | `databricks bundle run jobs_formula1_lakehouse_full_refresh` |
| Check status | `databricks bundle run --output JSON` or check Jobs UI |